# Informe de la Práctica: Pac-Man 
### Carla Carreras y Sandra Crevillen 

## Introducción y Objetivos

El objetivo principal de esta práctica es diseñar, entrenar y evaluar un agente inteligente capaz de resolver de forma óptima el entorno competitivo de Pac-Man. Para ello, se plantea la transición desde un enfoque clásico analítico y reactivo hacia una arquitectura avanzada que combina el aprendizaje profundo (*Deep Learning*) con algoritmos de búsqueda.


## Tarea 1 - Mejorar la función de evaluación con nuevas heurísticas

Dejando el código de la práctica tal y como nos lo clonamos del repositorio inicial, nos dimos cuenta que pac-man no se comportaba de la mejor manera para ganar el juego, no seguia una estrategia, ya que solo se centraba en ir hacia la comida más cercana, sin tener en cuenta la proximidad de los fantasmas. Para modificar esto, hemos implementado las dos heurísticas siguientes:

### 1. Heurística de Distancia a las Cápsulas de Poder
Las cápsulas modifican las reglas del juego, hacen que los fantasmas pasen de ser cazadores a presas, en vez de perseguir al pac-man, huyen como si tuviesen miedo. Si Pac-Man no sabe dónde están las cápsulas, puede pasar al lado de una y morir por no comérsela.

Las cápsulas en esta heuristica funcionan de la siguiente manera, si está muy lejos el valor de la heurística será muy bajo, entonces pac-man ignorará la cápsula y se centrará en comer los puntos que tiene al rededor. Si por el contrario, pac-man está muy cerca de la cápsula, el valor de la heurística es muy alto y por lo tanto la atracción hacia esa casilla es muchísimo mayor.

* **Ponderación asignada:** +10.0 / (distancia + 1). Le hemos puesto un peso bastante alto porque queremos que si pac-man tiene una cápsula cerca vaya directo a por ella sin rodeos.

### 2. Heurística de Seguridad contra Callejones (Acciones Legales)
El mayor problema del Pac-Man voraz es que calcula el beneficio a muy corto plazo y se mete en pasillos sin salida donde los fantasmas lo dejan atrapado y muere. Para solucionar esto, hemos decido premiar al agente en función del número de movimientos legales futuros (getLegalActions). Si un estado le deja con muchas salidas, es un estado seguro, pero si le deja con pocas opciones, como en un callejón, el valor baja.

* **Ponderación asignada:** +3.0 * len(state.getLegalActions(0)). Le hemos puesto un peso no muy alto pero constante para que el agente siempre prefiera las zonas abiertas del mapa antes que los pasillos cerrados o con pocas salidas.

## Tarea 2 - Entrenar el modelo con buenos juegos

Para que la red neuronal (PacmanNet) aprenda a jugar bien, primero necesitábamos entrenarlo dandole buenos ejemplos. Pusimos a jugar a nuestro Pacman mejorado con las heurísticas de la Tarea 1 durante varias partidas en el mapa clásico (mediumClassic). El agente funcionó bastante bien, logrando ganar un buen porcentaje de juegos y acumulando puntuaciones altas, en algunas partidas llegamos hata los 2000 puntos.

En total, hemos obtenido **48 archivos de partidas** (guardados como archivos .csv en la carpeta pacman_data), lo que nos dio un total de miles de ejemplos de jugadas con los que se ha alimentado, entrenado y validado la red neuronal.

La arquitectura implemnetada en PyTorch es una red neuronal artificial de tipo feedforward(MLP-Perceptron Multicapa) que esta compuesta por 3 capas lineales conectadas: 

1. **Capa de entrada:** Está oculta. Transforma el mapa aplanado en 256 neuronas con activación ReLU y Dropout del 30%

2. **Capa oculta:** También está oculta. Reduce a 128 neuronas con activación ReLU y Dropout del 30%

3. **Capa de salida:** Produce 5 valores, uno por accion posible (Stop, North, South, East, West)

### Ajuste de Hiperparámetros y Optimización
Para alcanzar el máximo rendimiento durante el entrenamiento de PacmanNet, hemos ajustado los siguientes parámetros en el script net.py:

1. **Planificador de Tasa de Aprendizaje Dinámico (StepLR):** Implementamos un planificador (scheduler) que reduce la tasa de aprendizaje (Learning Rate) a la mitad cada 30 épocas, partiendo de un valor inicial de 0.001 (LEARNING_RATE = 0.001). Esta estrategia permite que la red realice grandes ajustes algorítmicos al principio para mejores jugadas finalmente

2. **Regularización y Control de Sobreajuste:** Para evitar que la red simplemente memorizase de forma exacta las posiciones del mapa clásico (overfitting), utilizamos el algoritmo Adam. Lo combinamos estratégicamente con las capas de Dropout al 30% para forzar al modelo a generalizar la lógica del movimiento.

3. **Tamaño del lote (Batch Size):** Lo ajustamos a 64 para equilibrar la velocidad de procesamiento.

Al final de las 100 épocas (NUM_EPOCHS = 100), logramos que la red estabilice sus pérdidas (Loss) exportando el diccionario de pesos óptimos directamente al archivo models/pacman_model.pth.



## Tarea 3 - Implementar

Creamos una nueva clase de agente llamada exactamente AlphaBetaNeuralAgent,usando el algoritmo de **Poda Alpha-Beta** para calcular el futuro a profundidad 2 (simulando los movimientos de los fantasmas), pero en lugar de evaluar el tablero final con una función matemática simple, llamamos a la **Red Neuronal** que entrenamos en el paso anterior para que nos dé sus predicciones. Es decir, simulamos las miles de partidas imaginarias del futuro para ver qué pasará y cuando llega al final de la simulación, necesitamos la función matemática (Red Neuronal) para que le diga que la posición en la que estamos es buena y sume puntos o es mala y los reste.


La fórmula de puntuación final sigue la estructura combinada.
Esto lo usamos para hacer que el agente no se destruya y para ello no dejamos que la red haga todo, sino que la puntuación de cada decisión final se calcula sumando los tres componentes con pesos equilibrados:
final_score = play_score + (w_heuristic * trad_score) + (w_neural * neural_score)

- Predicción de la Red Neuronal(w_neuronal = 1.0): sabe por dónde suelen moverse los expertos para ganar y qué pasillos son más seguros.

- Heurística(w_heruristic = 2.0): penaliza al pac-man si se queda quieto y le premia si avanza hacia la comida,  le hemos puesto el doble del peso para asegurarnos de que el agente busque siempre limpiar el mapa para ganar la partida.

- Puntuación del juego: si en una jugada un fantasma nos come, el juego le pone un castigo de -500 puntos automáticamente, haciendo que Alpha-Beta descarte ese camino de inmediato, priorizando la supervivencia.

## Tarea 4 - Tabla Comparativa de Resultados y Conclusiones


| Configuración del Agente | Diseño Clásico (mediumClassic) <br>*(Media Score / Win Rate)* | Nuevo Diseño (customMaze) <br>*(Media Score / Win Rate)* |
| :--- | :---: | :---: |
| **Agente neural codicioso (sin modificaciones)** | 712.4 pts / 20% victorias | 342.1 pts / 0% victorias |
| **Agente neural codicioso + 2 nuevas heurísticas** | 1142.5 pts / 70% victorias | 910.4 pts / 40% victorias |
| **Alfa-Beta + red + heurística (final)** | **1685.0 pts / 100% victorias** | **1698.0 pts / 100% victorias** |





### Conclusiones del análisis
* El agente neuronal puro funciona aceptablemente en el mapa clásico, pero tiene un 0% de victorias en nuestro propio laberinto. Esto demuestra que las redes neuronales por sí solas no son optimas cuando las sacas del entorno exacto donde ha sido entrenada.

* **Las heurísticas:** Al meter las heurísticas de las cápsulas y los callejones, el agente voraz mejora notablemente en ambos laberintos. En CustomMaze, aunque la red neuronal seguía fallando, las heurísticas impidieron que se acorralara solo, subiendo la puntuación a 910.4 puntos y con alguna victoria.

* **Enfoque híbrido:** El mejor jugador ha sido AlphaBetaNeuralAgent. Al combinar la capacidad de predicción del futuro de Alpha-Beta (que sabe perfectamente qué movimientos matan o salvan al pac-man) con la evaluación inteligente de la red neuronal, hemos logrado ganar en el laberinto clásico con un **100% de victorias y 1685.0 puntos**. Además, ha sido el único capaz de mantenerse también en el mapa desconocido (CustomMaze) con un **100% de victorias y 1698.0 puntos**, demostrando que la búsqueda heurística compensa las debilidades y la falta de generalización de la red neuronal.

### 1. ¿Mejoró el rendimiento la incorporación de las nuevas heurísticas?
Sí, si comparamos el *Agente neuronal voraz puro* con el *Agente con las 2 nuevas heurísticas*, vemos que en el diseño clásico la puntuación media subió de **712.4 puntos a 1142.5 puntos**, logrando también más victorias (subiendo del 20% al 70%). 

Esto se debe a que la red neuronal por sí sola tiende a tomar decisiones basándose solo en patrones globales que ha memorizado, lo que a veces la hace caer en trampas. Al meterle nuestras dos heurísticas(ir a por las cápsulas si están cerca y huir de zonas con pocas acciones legales para no encerrarse en callejones), conseguimos que el Pac-Man juegue con mucha más seguridad, evitando muertes absurdas y alargando el tiempo de supervivencia.

### 2. ¿Y el algoritmo Alpha-Beta?
Alpha-Beta ha sido el que más nos ha hecho mejorar. Al combinar la poda Alpha-Beta con la evaluación de la red neuronal y nuestra estrategia fija y equilibrada de pesos, el rendimiento mejoro en el mapa clásico hasta un **100% de victorias y una media de 1685.0 puntos**.

La razón de este éxito es que el agente voraz solo reacciona al estado actual del tablero. En cambio, Alpha-Beta calcula el árbol de juego hacia el futuro (simulando los movimientos de los fantasmas). Esto le permite adelantarse a los peligros y, gracias a nuestro sistema de pesos, meter una marcha más y volverse agresivo para cazar a los fantasmas en cuanto se comen una cápsula. Ha demostrado ser, con diferencia, ser la que más alta puntuación da en las partidas.

### 3. ¿Se generaliza bien al nuevo diseño?
La red neuronal por sí sola no generaliza bien, pero el algoritmo híbrido (Alpha-Beta + Red) sí.

La red neuronal pura entrenada en mediumClassic se desorienta en el nuevo diseño debido al cambio en la disposición de los pasillos, con un **0% de victorias**. No sabe transferir lo aprendido a un entorno diferente.

Sin embargo, **Alpha-Beta híbrido consiguió mantener un 100% de victorias y una media de 1698.0 puntos** en el nuevo mapa desconocido. Esto demuestra que aunque la red neuronal le dé valores un poco distorsionados a las casillas por el cambio de mapa, Alpha-Beta calcula las consecuencias de sus movimientos a corto plazo, impidiendo que los fantasmas lo acorralen y permitiendo que el Pac-Man gane partidas en mapas que jamás ha visto.

## Tarea 5 - Análisis del rendimiento en customMaze

El nuevo mapa tiene aberturas donde el clásico tiene paredes, lo que lleva a generar caminos mucho más abiertos, cambiando por completo las distancias Manhattan y la dinámica tradicional del juego.

### 1. Comparativa de Resultados en customMaze
* **Agente Neuronal Voraz Puro:** Consiguió un **0% de victorias** (Media: 342.1 puntos). Al abrirse los muros, la red neuronal intentaba buscar pasillos basándose en el mapa clásico con el que fue entrenada. Al no existir esas referencias, Pac-Man se quedaba dando vueltas o chocando contra esquinas de forma ineficiente.

* **Agente Híbrido Alpha-Beta + Red Neuronal:** Logró un sólido **100% de victorias** (Media: **1698.0 pts**).

### 2. ¿Cómo afecta la "apertura" del mapa a la dinámica del juego?
En un mapa con caminos más abiertos, los fantasmas ya no tienen que rodear grandes bloques de paredes para perseguir a Pac-Man, ahora pueden cruzar por las nuevas aberturas para acorralarlo mucho más rápido. 

En esta situación, las distancias en línea recta (Manhattan) engañan por completo a las heurísticas simples. Sin embargo, nuestro agente de la Tarea 3 sobrevive y domina el mapa gracias a dos factores clave:

1. **La anticipación del árbol Alpha-Beta:** Al calcular a profundidad 2, el agente ve que el fantasma se va a colar por la nueva abertura del pasillo antes de que ocurra, cambiando su ruta de escape a tiempo.

2. **El equilibrio de nuestro sistema de pesos:** Dado que en un mapa abierto es más fácil cruzarse con los enemigos, la combinación que da prioridad a la heurística tradicional hace que Pac-Man aproveche de forma óptima los nuevos pasillos abiertos para avanzar limpiando las bolitas y arrinconar a los fantasmas vulnerables, lo que explica que hayamos alcanzado una puntuación tan alta (**1698.0 pts**).

### Conclusión Final 
* Las redes neuronales son herramientas muy potentes para evaluar situaciones complejas, pero tienen mucho  *overfitting* y falta de generalización cuando el entorno cambia 

* La búsqueda guiada por algoritmos clásicos como Alpha-Beta actúa como el motor de seguridad. No le importa si el mapa tiene formas nuevas, porque calcula las consecuencias de cada acción en el futuro inmediato. 

El éxito de nuestro agente final demuestra que combinar el cerebro de la red neuronal con los reflejos y la anticipación de la poda Alpha-Beta es la estrategia más robusta y eficiente.

## Comandos

La semilla aleatoria (seed) la hemos fijado directamente en el código fuente del proyecto. Por lo tanto, al ejecutar cualquiera de los comandos mostrados a continuación, el comportamiento de pac-man será idéntico, la misma partida y los mismos resultados obtenidos durante nuestras pruebas.

Para reproducir el comportamiento del modelo final guardado en `models/pacman_model.pth`, ejecuta los siguientes comandos en la terminal desde la raíz del proyecto:

#(mediumClassic) y (customMaze)
```bash
python3 pacman.py -p AlphaBetaNeuralAgent -a w_heuristic=2.0,w_neural=1.0,depth=2 -l mediumClassic

python3 pacman.py -p AlphaBetaNeuralAgent -a w_heuristic=2.0,w_neural=1.0,depth=2 -l customMaze

